# Lesson 5: Proximal Policy Optimization / PPO on CartPole

PPO is one of the most important practical policy-gradient algorithms.

It builds on actor-critic and advantages, but adds a constraint:

```text
do not let the policy change too much in one update
```

It does this with a clipped probability ratio.


## 1) Imports


In [ ]:
%pip install -U "gymnasium[classic-control]"

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)


## 2) Seeds and Device


In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3) Environments


In [ ]:
train_env = gym.make("CartPole-v1")
test_env = gym.make("CartPole-v1")

train_env.action_space.seed(SEED)
test_env.action_space.seed(SEED + 1)

state, info = train_env.reset(seed=SEED)
print("example state:", state)
print("observation space:", train_env.observation_space)
print("action space:", train_env.action_space)


## 4) Actor-Critic Network


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.actor = nn.Linear(hidden_dim, output_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return logits, value


## 5) Build Policy and Optimizer


In [ ]:
INPUT_DIM = train_env.observation_space.shape[0]
HIDDEN_DIM = 128
OUTPUT_DIM = train_env.action_space.n

policy = ActorCritic(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)

print(policy)


## 6) Collect One Episode

PPO stores the policy's old log probabilities when the data was collected.

Later, during updates, it compares the new policy to the old policy:

```text
ratio = exp(new_log_prob - old_log_prob)
```


In [ ]:
def collect_episode(env, policy, seed=None):
    policy.eval()

    states = []
    actions = []
    old_log_probs = []
    rewards = []
    values = []
    dones = []
    episode_reward = 0.0

    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            logits, value = policy(state_tensor)
            distribution = Categorical(logits=logits)
            action = distribution.sample()
            log_prob = distribution.log_prob(action)

        next_state, reward, terminated, truncated, info = env.step(action.item())
        done = terminated or truncated

        states.append(state_tensor.squeeze(0))
        actions.append(action.squeeze(0))
        old_log_probs.append(log_prob.squeeze(0))
        values.append(value.squeeze(0))
        rewards.append(reward)
        dones.append(float(done))
        episode_reward += reward
        state = next_state

    states = torch.stack(states)
    actions = torch.stack(actions)
    old_log_probs = torch.stack(old_log_probs)
    values = torch.stack(values)

    return states, actions, old_log_probs, rewards, values, dones, episode_reward


## 7) Returns and Advantages

For this beginner PPO version, we use Monte Carlo returns and simple advantages:

```text
advantage = return - value
```

Many production PPO implementations use GAE here.


In [ ]:
def calculate_returns(rewards, discount_factor):
    returns = []
    running_return = 0.0

    for reward in reversed(rewards):
        running_return = reward + discount_factor * running_return
        returns.insert(0, running_return)

    return torch.as_tensor(returns, dtype=torch.float32, device=device)


def calculate_advantages(returns, values, normalize=True):
    advantages = returns - values.detach()

    if normalize and len(advantages) > 1:
        std = advantages.std(unbiased=False)
        if std > 1e-8:
            advantages = (advantages - advantages.mean()) / (std + 1e-8)

    return advantages


## 8) PPO Update

PPO compares new action probabilities against old action probabilities.

```text
ratio = new_prob / old_prob
```

Using logs:

```python
ratio = torch.exp(new_log_prob - old_log_prob)
```

Then it clips the ratio so one update cannot push the policy too far.


In [ ]:
def update_policy(policy, states, actions, old_log_probs, advantages, returns, optimizer, ppo_steps, ppo_clip):
    policy_losses = []
    value_losses = []

    old_log_probs = old_log_probs.detach()
    advantages = advantages.detach()
    returns = returns.detach()

    for _ in range(ppo_steps):
        logits, values = policy(states)
        distribution = Categorical(logits=logits)
        new_log_probs = distribution.log_prob(actions)
        entropy = distribution.entropy().mean()

        ratio = torch.exp(new_log_probs - old_log_probs)
        unclipped = ratio * advantages
        clipped = torch.clamp(ratio, 1.0 - ppo_clip, 1.0 + ppo_clip) * advantages

        policy_loss = -torch.min(unclipped, clipped).mean()
        value_loss = F.mse_loss(values, returns)
        loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        policy_losses.append(policy_loss.item())
        value_losses.append(value_loss.item())

    return np.mean(policy_losses), np.mean(value_losses)


## 9) Train One Episode With PPO


In [ ]:
def train_one_episode(env, policy, optimizer, discount_factor, ppo_steps, ppo_clip, seed=None):
    states, actions, old_log_probs, rewards, values, dones, episode_reward = collect_episode(
        env, policy, seed=seed
    )

    returns = calculate_returns(rewards, discount_factor)
    advantages = calculate_advantages(returns, values)

    policy_loss, value_loss = update_policy(
        policy,
        states,
        actions,
        old_log_probs,
        advantages,
        returns,
        optimizer,
        ppo_steps,
        ppo_clip,
    )

    return policy_loss, value_loss, episode_reward


## 10) Evaluate


In [ ]:
def evaluate(env, policy, seed=None):
    policy.eval()

    episode_reward = 0.0
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()

        state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

    return episode_reward


## 11) Training Loop


In [ ]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
PPO_STEPS = 5
PPO_CLIP = 0.2
N_TRIALS = 25
REWARD_THRESHOLD = 475
PRINT_EVERY = 10

train_rewards = []
test_rewards = []
policy_losses = []
value_losses = []
recent_test_rewards = deque(maxlen=N_TRIALS)

for episode in range(1, MAX_EPISODES + 1):
    policy_loss, value_loss, train_reward = train_one_episode(
        train_env,
        policy,
        optimizer,
        DISCOUNT_FACTOR,
        PPO_STEPS,
        PPO_CLIP,
        seed=SEED + episode,
    )
    test_reward = evaluate(test_env, policy, seed=SEED + 10_000 + episode)

    train_rewards.append(train_reward)
    test_rewards.append(test_reward)
    policy_losses.append(policy_loss)
    value_losses.append(value_loss)
    recent_test_rewards.append(test_reward)

    if episode % PRINT_EVERY == 0:
        print(
            f"| Episode: {episode:3} | "
            f"Mean Train: {np.mean(train_rewards[-N_TRIALS:]):6.1f} | "
            f"Mean Test: {np.mean(recent_test_rewards):6.1f} | "
            f"Policy Loss: {policy_loss:8.3f} | Value Loss: {value_loss:8.3f} |"
        )

    if len(recent_test_rewards) == N_TRIALS and np.mean(recent_test_rewards) >= REWARD_THRESHOLD:
        print(f"Reached reward threshold in {episode} episodes")
        break


## 12) Plot Rewards


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train_rewards, label="Train Reward", alpha=0.7)
plt.plot(test_rewards, label="Test Reward")
plt.axhline(REWARD_THRESHOLD, color="red", linestyle="--", label="Threshold")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()


## 13) Watch Policy


In [ ]:
def watch_policy(policy, seed=SEED, max_steps=500):
    render_env = gym.make("CartPole-v1", render_mode="human")
    state, info = render_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated) and steps < max_steps:
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()
        state, reward, terminated, truncated, info = render_env.step(action)
        total_reward += reward
        steps += 1

    print(f"Episode finished. Total reward: {total_reward}, steps: {steps}")
    render_env.close()


# Uncomment after training if your machine supports GUI rendering.
# watch_policy(policy)


## 14) Exercises

1. Change `PPO_CLIP` from `0.2` to `0.05` and `0.4`. What happens?
2. Set `PPO_STEPS = 1`. How is this closer to A2C?
3. Print `ratio.min()`, `ratio.mean()`, and `ratio.max()` during update.

## 15) Mental Model

PPO says:

```text
Use the data more than once, but do not let the new policy move too far away from the old policy.
```

That is why it is stable and widely used in robotics simulators like Isaac Lab.
